# DarkPipe 0.10 — shadows observables e inobservables condicionados

Este Colab verifica el resultado oficial ya congelado de `DP-OBS-SHADOW-INOBS-0.10-20260826`. **No relanza la campaña histórica**. Descarga el release compacto, comprueba hashes y reproduce únicamente la adjudicación descriptiva de los perfiles derivados.

Jurisdicción: aceleración efectiva firmada y masa encerrada esférico-equivalente condicionadas por dinámica circular newtoniana y nuisances declarados. No identifica partículas, densidad 3D, MOND, Lambda-CDM, mecanismo gravitatorio ni hiperestados plasmáticos.

In [ ]:
!git clone --depth 1 --branch v0.10.0 https://github.com/FacundoFirmenich/darkpipe-realdata.git /content/darkpipe-realdata
%cd /content/darkpipe-realdata
!python -m pip install -q -e .

In [ ]:
import hashlib, json
from pathlib import Path

root = Path('evidence/v010_shadow_inobservable')
expected_sha256 = {
    'derived_inobservable_profiles.csv': 'aee55c110eb5dbe593a37e173633383c77f46251527678fc188d8ff4ce6e0977',
    'inobservable_summary.json': 'cc5871df311a03baefec370da354bbc04b6042c2a8dae7d1431748e576378333',
    'manifest.json': '798672ea50f76cfa9ae54a48ec4726154a1320ff1e50f875b4e9ff0bae6b2062',
    'observable_shadow_inobservable.png': '48c7b6bf271fdd8a06c9c960d260ccd4ab4f123730b6b30e3ba75c405931d8d8',
    'SUBSTANTIVE_CLOSURE_ES.md': '5354f591861973ae20d874cb3c8e7c7c6254960bc6167e8853c4aa8222577a34',
}
for name, expected in expected_sha256.items():
    observed = hashlib.sha256((root / name).read_bytes()).hexdigest()
    assert observed == expected, (name, observed, expected)

summary = json.loads((root / 'inobservable_summary.json').read_text(encoding='utf-8'))
assert summary['decision'] == 'DERIVED_CONDITIONAL_INOBSERVABLE_PROFILES_AVAILABLE'
assert summary['authority'] == 'DERIVED_EFFECTIVE_INOBSERVABLE_CONDITIONAL_NOT_ONTOLOGIZED'
assert summary['selection']['galaxies'] == 149
assert summary['selection']['points'] == 2700
print('Hashes: PASS')
print('Decisión:', summary['decision'])
print('Estados:', summary['status_counts'])

In [ ]:
import numpy as np
import pandas as pd

profiles = pd.read_csv(root / 'derived_inobservable_profiles.csv')
assert len(profiles) == 2700
assert profiles['galaxy'].nunique() == 149
assert profiles.duplicated(['galaxy', 'radius_nominal_kpc']).sum() == 0
assert np.isfinite(profiles.select_dtypes(include=[np.number]).to_numpy()).all()

status = profiles['inobservable_status'].value_counts()
assert status['POSITIVE_SIGNED_PROFILE_SUPPORTED_95'] == 1917
assert status['SIGN_AMBIGUOUS_95'] == 775
assert status['NEGATIVE_SIGNED_PROFILE_SUPPORTED_95'] == 8

by_galaxy = profiles.groupby('galaxy')['inobservable_status'].value_counts().unstack(fill_value=0)
positive = 'POSITIVE_SIGNED_PROFILE_SUPPORTED_95'
print('Galaxias con algún radio positivo:', int((by_galaxy[positive] > 0).sum()), '/ 149')
print('Mediana muestral g_I:', profiles['g_inobservable_p50_m_s2'].median(), 'm s^-2')
print('Mediana muestral M_I:', profiles['mass_inobservable_p50_solar'].median(), 'masas solares')

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(root / 'observable_shadow_inobservable.png')))

## Interpretación sustantiva

Ya existe el objeto exigido por la misión: inobservables radiales derivados de una shadow cinemático-bariónica real, con incertidumbre y signo preservados. La fuerte presencia de perfiles positivos, especialmente en los radios externos, es compatible con una contribución dinámica adicional; no determina su ontología. Los 775 radios ambiguos y los 8 negativos sostenidos son fronteras informativas, no descartes.

El siguiente paso científico es una inferencia multi-shadow con al menos una proyección independiente —por ejemplo lensing— y covarianza jerárquica por galaxia. Solo después corresponde comparar qué arquitecturas físicas abarcan simultáneamente la cinemática, la geometría y los casos adversos.

In [ ]:
import zipfile
from google.colab import files

out = Path('/content/darkpipe_v010_compact_result.zip')
with zipfile.ZipFile(out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for source in [
        *sorted(root.iterdir()),
        Path('docs/PREREGISTRATION_OBSERVABLE_SHADOW_INOBSERVABLE_0.10.md'),
        Path('docs/V010_SUBSTANTIVE_ADJUDICATION_ES_2026-08-26.md'),
        Path('LICENSE'),
    ]:
        zf.write(source)
print(out, out.stat().st_size, 'bytes')
files.download(str(out))